# Chapter 12 — Precision in Training

Reproduces every measurement in Chapter 12. The underflow run fits one fixed batch with this
course's model (623,744 parameters) and, along the way, casts every gradient to fp16 for real.
It takes a minute or two on a CPU.

In [ ]:
import math, sys, pathlib, codecs, contextlib, io as _io
with contextlib.redirect_stdout(_io.StringIO()):
    import this                      # the Zen of Python prints itself on import
import torch
root = pathlib.Path.cwd()
while not (root / "tfs").is_dir() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root))
from tfs.model import Config, Model

## 12.3 — the share of gradients fp16 deletes, as the model converges

In [ ]:
torch.manual_seed(0)
# Fixed, version-stable training text: the Zen of Python, from the standard library.
text = (codecs.decode(this.s, "rot13") * 4).encode("utf-8")
cfg = Config(vocab_size=256, d_model=128, n_layer=4, n_head=4, n_kv_head=2, d_head=32, d_ff=256, max_T=256)
model = Model(cfg)
n_params = sum(p.numel() for p in model.parameters()); print("params", n_params); assert n_params == 623_744
B, T = 8, 128
x = torch.tensor([list(text[i * T:(i + 1) * T]) for i in range(B)], dtype=torch.long)
opt = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=0)

rows = {}
for step in range(1201):
    opt.zero_grad(set_to_none=True)
    _, loss, _ = model(x, targets=x)
    loss.backward()
    if step in (0, 100, 300, 600, 1200):
        g = torch.cat([p.grad.flatten() for p in model.parameters() if p.grad is not None])
        g = g[g != 0]; a = g.abs(); hh = g.half()
        zero = (hh == 0).float().mean().item()
        sub = ((a < 2 ** -14) & (hh != 0)).float().mean().item()
        below = (a < 2 ** -24).float().mean().item()
        scaled = {S: ((g * S).half() == 0).float().mean().item() for S in (1, 128, 1024, 32768)}
        over = {S: torch.isinf((g * S).half()).float().mean().item() for S in (1, 128, 1024, 32768)}
        rows[step] = dict(loss=loss.item(), med=a.median().item(), zero=zero, sub=sub, below=below,
                          scaled=scaled, over=over)
        print(f"step {step:4d}  loss {loss.item():.4g}  median|g| {a.median().item():.3e}  "
              f"-> zero {zero:.2%}  subnormal {sub:.1%}  (below 2^-24: {below:.2%})")
    opt.step()

r0, rN = rows[0], rows[1200]
assert r0["zero"] < 0.001, "an fp16 run looks fine at initialisation"
assert rN["zero"] > 0.2, "and loses a large share of its gradients by convergence"
assert all(rows[a]["zero"] <= rows[b]["zero"] for a, b in [(0, 300), (300, 600), (600, 1200)]), "monotone"
assert rN["below"] > rN["zero"], "counting against 2^-24 overstates: (2^-25, 2^-24) rounds UP"
print("\nloss scale  vanish   overflow   (at step 1200)")
for S in (1, 128, 1024, 32768):
    print(f"{S:10d}  {rN['scaled'][S]:7.2%}  {rN['over'][S]:7.2%}")
print(f"exact at 32768: vanish {rN['scaled'][32768]:.2e}  overflow {rN['over'][32768]:.2e}")
assert rN["scaled"][32768] < 1e-4 and rN["over"][32768] < 1e-4

## 12.2 — the same matmul in three formats

In [ ]:
torch.manual_seed(0)
A = torch.randn(512, 512, dtype=torch.float64); Bm = torch.randn(512, 512, dtype=torch.float64)
ref = A @ Bm
err = {}
for dt in (torch.float32, torch.float16, torch.bfloat16):
    # inputs and output rounded to the format, products accumulated in fp32: what a tensor core does
    out = (A.to(dt).float() @ Bm.to(dt).float()).to(dt).double()
    err[dt] = ((out - ref).abs().max() / ref.abs().max()).item()
    print(f"{str(dt):16s} max relative error {err[dt]:.3g}")
print(f"bf16 / fp16 = {err[torch.bfloat16] / err[torch.float16]:.1f}x")
assert err[torch.float16] < err[torch.bfloat16]

## 12.5 — the accumulator matters more than the data

In [ ]:
torch.manual_seed(0)
x = torch.rand(4096, dtype=torch.float64) * 0.01
exact = x.sum().item()
acc = torch.tensor(0.0, dtype=torch.float16)
for v in x.half():                  # a genuine fp16 running total
    acc = acc + v
rel = lambda v: abs(v - exact) / exact
f16 = acc.item(); f32 = x.half().sum(dtype=torch.float32).item(); trap = x.half().sum(dtype=torch.float16).item()
print(f"exact {exact:.10f}")
print(f"fp32 values, fp32 accumulate  {x.float().sum().item():.10f}  rel {rel(x.float().sum().item()):.2g}")
print(f"fp16 values, fp16 accumulate  {f16:.10f}  rel {rel(f16):.2g}")
print(f"fp16 values, fp32 accumulate  {f32:.10f}  rel {rel(f32):.2g}   ({rel(f16) / rel(f32):,.0f}x better)")
print(f"the trap: x.half().sum(dtype=float16) = {trap}  rel {rel(trap):.2g} -- not an fp16 accumulator")
assert rel(f16) > 0.05 and rel(f32) < 1e-5 and rel(trap) < 1e-3

## Two edge cases the chapter states

In [ ]:
assert torch.tensor(3.4e38).to(torch.bfloat16).item() == float("inf")     # bf16 CAN overflow, at the very top
assert torch.tensor(65519.).half().item() == 65504 and torch.tensor(65520.).half().item() == float("inf")
assert torch.tensor(1e-7).half().item() == 2 * 2 ** -24                    # 19% error, in the subnormals
print("ok")